In [1]:
import os, torch
print("working folder:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

working folder: /content
GPU: Tesla T4


In [2]:
%cd /content
!git clone https://github.com/samyuktaathreya/pinyin_audio_recognizer_model.git repo
%cd /content/repo/pinyin_scorer
%pip install -q pypinyin

/content
Cloning into 'repo'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 90 (delta 0), reused 2 (delta 0), pack-reused 83 (from 1)
Receiving objects: 100% (90/90), 27.78 MiB | 17.18 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/repo/pinyin_scorer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 50.9 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/pinyin_runs

Mounted at /content/drive
stage1	stage2_probs.csv  stage4   stage5   stage6b
stage2	stage3		  stage4b  stage6a  stage6b_probs.csv


In [5]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo log --oneline -1
!ls *.py
!grep -c "Hiss only" data.py

/content/repo/pinyin_scorer
966aafe (HEAD -> main, origin/main, origin/HEAD) updated train.py
data.py        eval_sources.py	   pinyin_units.py     prepare_syllables.py
diagnose.py    find_duplicates.py  prepare_aishell.py  score.py
dump_probs.py  model.py		   prepare_folders.py  train.py
0


In [7]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/ext
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl
!python prepare_folders.py --root /content/ext/standard --out-dir manifests/folders

train: 6560 clips -> manifests/syl_train.jsonl
test: 3279 clips -> manifests/syl_test.jsonl
clips per speaker: {'FV1': 1640, 'FV2': 1640, 'FV3': 1639, 'MV1': 1640, 'MV2': 1640, 'MV3': 1640}
davinfifield     train   1617 clips
digmandarin      train   1627 clips
gitcode          test    1283 clips
hotoo            train   1616 clips
hugolpz          train   1656 clips  (4 skipped)
losttheory       train   1568 clips
mandarintemple   train   1603 clips
misc             train    168 clips
sreading         train   1616 clips
studycli         test    1572 clips
uvm              test    1300 clips
yabla            train   1628 clips
yoyochinese      train   1612 clips

train: 14711 clips (4.0 h)   test: 4155 clips


In [ ]:
!grep -c "Hiss only" data.py
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

In [ ]:
!ps aux | grep "train.py" | grep -v grep | wc -l
!ls /content/repo/pinyin_scorer/manifests /content/ext 2>&1 | head -5
!ls /content/drive/MyDrive/pinyin_runs/stage4/last

In [ ]:
%cd /content/repo/pinyin_scorer

In [ ]:
!setsid nohup python -u train.py --init-from /content/drive/MyDrive/pinyin_runs/stage4/last \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4b \
    --epochs 5 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 100 --augment --workers 2 \
    > /content/train_stage4b.log 2>&1 &

In [ ]:
!tail -3 /content/train_stage4b.log
!grep "^epoch [0-9]* initial" /content/train_stage4b.log

In [ ]:
!python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/stage4b/best \
    --manifests manifests/syl_test.jsonl manifests/folders/test_*.jsonl --out stage4_heldout.csv

In [ ]:
!ps aux | grep "train.py" | grep -v grep | grep -o "\-\-out [^ ]*" | sort -u

In [ ]:
import json
for line in open("/content/drive/MyDrive/pinyin_runs/stage4/log.jsonl"):
    d = json.loads(line)
    print(d["epoch"], *(f"{d[k]:.3f}" for k in
          ["syllable_initial_acc", "syllable_final_acc", "syllable_tone_acc", "syllable_all3_acc"]))

In [ ]:
!pkill -f train.py
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "no-augment-sources" prepare_folders.py; grep -c "duplicate_pairs" find_duplicates.py; grep -c "def speech_power" data.py

In [10]:
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/std
!ls /content/std/standard

davinfifield  hotoo	    losttheory	    sreading  yabla
digmandarin   hugolpz	    mandarintemple  studycli  youtube_man
gitcode       left_out.csv  misc	    uvm       yoyochinese


In [12]:
ROOT = "/content/std/standard"
!python find_duplicates.py {ROOT}/*/

read  1617 syllables from davinfifield
read  1627 syllables from digmandarin
read  1283 syllables from gitcode
read  1616 syllables from hotoo
read  1660 syllables from hugolpz
read  1568 syllables from losttheory
read  1603 syllables from mandarintemple
read   168 syllables from misc
read  1616 syllables from sreading
read  1572 syllables from studycli
read  1300 syllables from uvm
read  1628 syllables from yabla
read     0 syllables from youtube_man
read  1612 syllables from yoyochinese

source A         source B          shared  same length  same recording?
gitcode          sreading            1283          93%              90%   <-- check
davinfifield     sreading            1594          17%              17%
losttheory       uvm                 1244          14%              11%
misc             sreading             165          10%               2%
sreading         yabla               1608          15%               1%
misc             yoyochinese          168          11%       

In [13]:
import pandas as pd
from IPython.display import Audio, display
ex = pd.read_csv("duplicate_examples.csv")
pair = ex[(ex.source_a == "losttheory") & (ex.source_b == "uvm")]
for _, r in pair.sample(3, random_state=0).iterrows():
    print(f"\n{r.syllable}  (similarity {r.similarity})")
    display(Audio(r.file_a)); display(Audio(r.file_b))


dao1  (similarity 0.986)



ang4  (similarity 0.978)



weng1  (similarity 0.976)


In [14]:
ROOT = "/content/std/standard"
!python prepare_folders.py --root {ROOT} --out-dir manifests/std \
    --test-sources gitcode studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory
!ls manifests/syl_train.jsonl manifests/syl_test.jsonl

davinfifield     train   1617 clips
digmandarin      train   1627 clips
gitcode          test    1283 clips
hotoo            train   1616 clips
hugolpz          train   1656 clips  (4 skipped)
mandarintemple   train   1603 clips
misc             train    168 clips
studycli         test    1572 clips
uvm              test    1300 clips  (no added noise)
yabla            train   1628 clips
yoyochinese      train   1612 clips

train: 11527 clips (3.1 h)   test: 4155 clips
manifests/syl_test.jsonl  manifests/syl_train.jsonl


In [ ]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl

In [ ]:
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/std/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage5 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

In [ ]:
for name in ["stage2", "stage5"]:
    print(f"\n######## {name}")
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/std/test_*.jsonl --out {name}_std.csv
!cp stage2_std.csv stage5_std.csv /content/drive/MyDrive/

In [ ]:
import json, os, glob, collections
train = [json.loads(l) for f in ["manifests/syl_train.jsonl", "manifests/std/train.jsonl"] for l in open(f)]
per_speaker = collections.Counter(r["speaker"] for r in train)
print(len(train), "training clips:", dict(per_speaker))
train_paths = {os.path.realpath(r["audio"]) for r in train}
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    speakers = {r["speaker"] for r in rows}
    same_files = sum(os.path.realpath(r["audio"]) in train_paths for r in rows)
    print(f"{tf}: {len(rows)} clips, speakers {sorted(speakers)}, "
          f"speakers also in training: {sorted(speakers & set(per_speaker)) or 'none'}, "
          f"identical files in training: {same_files}")

In [ ]:
import re
from pathlib import Path
dst = Path("/content/tp_by_speaker")
for f in Path("/content/tone_perfect").rglob("*.mp3"):
    m = re.match(r"^[a-zü:v]+[1-5]_([A-Za-z0-9]+)", f.name)
    if m:
        d = dst / f"tp_{m.group(1)}"
        d.mkdir(parents=True, exist_ok=True)
        if not (d / f.name).exists():
            (d / f.name).symlink_to(f)
!python find_duplicates.py /content/tp_by_speaker/*/ /content/std/standard/*/ \
    --out tp_dup_examples.csv --summary tp_dup_pairs.csv

In [ ]:
import json, glob, random
from IPython.display import Audio, display
from score import Scorer
scorer = Scorer("/content/drive/MyDrive/pinyin_runs/stage5/best")
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    scored = []
    for r in random.Random(0).sample(rows, min(40, len(rows))):
        res = scorer.score(r["audio"], r["pinyin"])
        scored.append((min(p["score"] for p in res["parts"].values()), r, res["parts"]))
    scored.sort(key=lambda x: x[0])
    print(f"\n===== {tf}")
    for label, group in [("WORST", scored[:3]), ("BEST", scored[-3:])]:
        for conf, r, p in group:
            print(f"{label}: target {r['pinyin']:7} heard {p['initial']['heard']} + {p['final']['heard']} "
                  f"+ tone {p['tone']['heard']}   (weakest part {conf:.2f})")
            display(Audio(r["audio"]))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "def normalize_loudness" data.py; grep -c "duplicate_pairs" find_duplicates.py

In [ ]:
for name in ["stage2", "stage5"]:
    print(f"\n######## {name}")
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/std/test_*.jsonl --out {name}_norm.csv

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "duplicate_pairs" find_duplicates.py

In [ ]:
import json, os, glob, collections
train = [json.loads(l) for f in ["manifests/syl_train.jsonl", "manifests/std/train.jsonl"] for l in open(f)]
per_speaker = collections.Counter(r["speaker"] for r in train)
print(len(train), "training clips:", dict(per_speaker))
train_paths = {os.path.realpath(r["audio"]) for r in train}
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    speakers = {r["speaker"] for r in rows}
    same_files = sum(os.path.realpath(r["audio"]) in train_paths for r in rows)
    print(f"{tf}: {len(rows)} clips, speakers {sorted(speakers)}, "
          f"speakers also in training: {sorted(speakers & set(per_speaker)) or 'none'}, "
          f"identical files in training: {same_files}")

In [ ]:
import re
from pathlib import Path
dst = Path("/content/tp_by_speaker")
for f in Path("/content/tone_perfect").rglob("*.mp3"):
    m = re.match(r"^[a-zü:v]+[1-5]_([A-Za-z0-9]+)", f.name)
    if m:
        d = dst / f"tp_{m.group(1)}"
        d.mkdir(parents=True, exist_ok=True)
        if not (d / f.name).exists():
            (d / f.name).symlink_to(f)
!python find_duplicates.py /content/tp_by_speaker/*/ /content/std/standard/*/ \
    --out tp_dup_examples.csv --summary tp_dup_pairs.csv

In [ ]:
import pandas as pd
from IPython.display import Audio, display

df = pd.read_csv("stage2_norm.csv")
g = df[df.source == "test_gitcode"]
print(len(g), "gitcode clips the model got wrong\n")
for _, r in g.sample(min(12, len(g)), random_state=1).iterrows():
    wrong = [h for h in ("initial", "final", "tone") if r[f"wrong_{h}"]]
    print(f"target {r.target:8} heard {r.heard_initial} + {r.heard_final} + tone {r.heard_tone}"
          f"   (wrong: {', '.join(wrong)})")
    display(Audio(r.audio))

In [ ]:
%cd /content/repo/pinyin_scorer

In [ ]:
import base64, io, random
import numpy as np, soundfile as sf
from IPython.display import HTML, display
from data import load_audio, SAMPLE_RATE

def clip_tag(path):
    """Audio player for one clip: volume evened out, silence trimmed off the ends."""
    wav = load_audio(path)
    n = len(wav) // 160
    rms = np.sqrt((wav[:n * 160].reshape(n, 160) ** 2).mean(axis=1))
    active = np.where(rms > 0.05 * rms.max())[0]
    if len(active):
        wav = wav[max(0, active[0] * 160 - 3200): (active[-1] + 1) * 160 + 3200]
    buf = io.BytesIO()
    sf.write(buf, wav, SAMPLE_RATE, format="WAV", subtype="PCM_16")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<audio controls preload="none" style="width:110px" src="data:audio/wav;base64,{b64}"></audio>'

def grid(kind="final", speakers=None):
    idx, units = (1, FINALS) if kind == "final" else (0, INITIALS)
    speakers = speakers or sorted(by_speaker)
    rng = random.Random(0)
    head = "<tr><th></th>" + "".join(f"<th>{s}</th>" for s in speakers) + "</tr>"
    rows = []
    for unit in units:
        cells = [f"<th>{unit}</th>"]
        for s in speakers:
            cands = [(p, r) for p, r in by_speaker[s] if p[idx] == unit]
            if not cands:
                cells.append("<td>–</td>")
                continue
            p, r = rng.choice([c for c in cands if c[0][2] == "1"] or cands)
            cells.append(f"<td style='font-size:11px'>{r['pinyin']}<br>{clip_tag(r['audio'])}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f"<div style='overflow-x:auto'><table>{head}{''.join(rows)}</table></div>"))

web = ["davinfifield", "digmandarin", "gitcode", "hotoo", "hugolpz", "mandarintemple",
       "misc", "studycli", "uvm", "yabla", "yoyochinese"]
grid("final", web)

In [ ]:
grid("initial", web)

In [ ]:
checks = [("digmandarin", "lan"), ("digmandarin", "shang"), ("digmandarin", "zhong"),
          ("hugolpz", "nao"), ("hugolpz", "fan"), ("hugolpz", "meng"),
          ("gitcode", "gong"), ("davinfifield", "miao"), ("davinfifield", "weng")]
for spk, base in checks:
    for p, r in by_speaker[spk]:
        if r["pinyin"][:-1] == base:
            print(f"{spk:14} {r['pinyin']:8} {r['audio'].split('/')[-1]}")
            display(Audio(r["audio"]))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!python prepare_folders.py --root /content/std/standard --out-dir manifests/good \
    --test-sources studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory davinfifield digmandarin gitcode hugolpz yoyochinese

In [ ]:
import json, glob, random
import soundfile as sf
from IPython.display import Audio, display

rows = [json.loads(l) for m in ["manifests/good/train.jsonl", *glob.glob("manifests/good/test_*.jsonl")]
        for l in open(m)]
speakers = ["hotoo", "mandarintemple", "misc", "studycli", "uvm", "yabla"]
rng = random.Random()          # different clips every run; use random.Random(0) to repeat a set

for spk in speakers:
    print(f"\n==================== {spk}")
    mine = [r for r in rows if r["speaker"] == spk]
    for tone in "1234":
        pool = [r for r in mine if r["pinyin"].endswith(tone)]
        for r in rng.sample(pool, min(2, len(pool))):
            wav, sr = sf.read(r["audio"])
            print(f"{r['pinyin']:8} {r['audio'].split('/')[-1]}")
            display(Audio(wav.T if wav.ndim > 1 else wav, rate=sr))

In [ ]:
import json, glob, random, base64, io, collections
import soundfile as sf
from IPython.display import HTML, display
from pinyin_units import split_syllable, INITIALS, FINALS
from data import load_audio, SAMPLE_RATE

by_speaker = collections.defaultdict(list)
for m in ["manifests/good/train.jsonl", *glob.glob("manifests/good/test_*.jsonl")]:
    for line in open(m):
        r = json.loads(line)
        parts = split_syllable(r["pinyin"])
        if parts:
            by_speaker[r["speaker"]].append((parts, r))

def clip_tag(path):
    buf = io.BytesIO()
    sf.write(buf, load_audio(path), SAMPLE_RATE, format="WAV", subtype="PCM_16")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<audio controls preload="none" style="width:110px" src="data:audio/wav;base64,{b64}"></audio>'

def grid(kind="initial", speakers=None, seed=None):
    idx, units = (0, INITIALS) if kind == "initial" else (1, FINALS)
    speakers = speakers or sorted(by_speaker)
    rng = random.Random(seed)
    head = "<tr><th></th>" + "".join(f"<th>{s}</th>" for s in speakers) + "</tr>"
    rows = []
    for unit in units:
        cells = [f"<th>{unit}</th>"]
        for s in speakers:
            cands = [r for p, r in by_speaker[s] if p[idx] == unit]
            if not cands:
                cells.append("<td>–</td>")
                continue
            r = rng.choice(cands)
            cells.append(f"<td style='font-size:11px'>{r['pinyin']}<br>{clip_tag(r['audio'])}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f"<div style='overflow-x:auto'><table>{head}{''.join(rows)}</table></div>"))

grid("initial", ["hotoo", "mandarintemple", "misc", "studycli", "uvm", "yabla"])

In [ ]:
!git -C /content/repo pull

In [ ]:
!python dump_probs.py --model /content/drive/MyDrive/pinyin_runs/stage2/best \
    --manifests manifests/syl_test.jsonl manifests/good/train.jsonl manifests/good/test_*.jsonl \
    --out /content/drive/MyDrive/pinyin_runs/stage2_probs.csv

In [ ]:
import pandas as pd
from IPython.display import Audio, display

CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
SOURCES = ["hotoo", "misc"]
SYLLABLES = ["zhi", "chi", "shi", "ri", "zi", "ci", "si"]

df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
df = df[df.source.isin(SOURCES) & df.target.str[:-1].isin(SYLLABLES)].copy()

def heard(row, part):   # the model's top guess for one part, ignoring silence
    cols = [c for c in df.columns if c.startswith(part + ":") and "silence" not in c]
    return max(cols, key=lambda c: float(row[c])).split(":")[1]

for part in ["initial", "final", "tone"]:
    df["heard_" + part] = df.apply(heard, axis=1, part=part)
df["wrong"] = (df.heard_initial != df.target_initial) | (df.heard_final != df.target_final)
df = df.sort_values(["wrong", "source", "target"], ascending=[False, True, True])

print(f"{len(df)} clips, {df.wrong.sum()} where the model heard a different initial or final\n")
for _, r in df.iterrows():
    mark = "WRONG" if r.wrong else "ok   "
    print(f"{mark}  {r.source:6} {r.target:5}  model heard: {r.heard_initial} + {r.heard_final} + tone {r.heard_tone}")
    display(Audio(r.audio))

In [7]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull

In [4]:
# Run A: clean data only
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage6a \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 200 --augment --workers 2

In [8]:
# Run B: the same, plus the two changes
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage6b \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 200 --augment --workers 2 \
    --voice-augment --per-head-mix

In [9]:
for name in ["stage2", "stage6a", "stage6b"]:
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl --out {name}_good.csv

In [10]:
!python dump_probs.py --model /content/drive/MyDrive/pinyin_runs/stage6b/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --out /content/drive/MyDrive/pinyin_runs/stage6b_probs.csv

In [16]:
import pandas as pd
from IPython.display import Audio, display

CSV = "/content/drive/MyDrive/pinyin_runs/stage6b_probs.csv"
SOURCE = "uvm"       # "uvm", "studycli", "FV3", "MV3", or None for all
PART   = "final"     # "initial", "final" or "tone"
TARGET = "eng"       # what the clip is labelled
HEARD  = "ang"       # what the model heard instead, or None for any wrong answer
LIMIT  = 30          # most clips to play

df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
cols = [c for c in df.columns if c.startswith(PART + ":") and "silence" not in c]
probs = df[cols].astype(float)
df["heard"] = probs.idxmax(axis=1).str.split(":").str[1]
df["p_target"] = [2.718281828 ** probs.at[i, f"{PART}:{t}"] for i, t in zip(df.index, df["target_" + PART])]

m = (df["target_" + PART] == TARGET) & (df.heard != TARGET)
if HEARD:  m &= df.heard == HEARD
if SOURCE: m &= df.source == SOURCE
hits = df[m].sort_values("p_target")

print(f"{len(hits)} clips labelled {TARGET} where the model heard {HEARD or 'something else'}\n")
for _, r in hits.head(LIMIT).iterrows():
    print(f"{r.source:9} {r.target:8} heard {r.heard:5} target's probability {r.p_target:.1%}")
    display(Audio(r.audio))

32 clips labelled eng where the model heard ang

uvm       sheng3   heard ang   target's probability 0.1%


uvm       feng3    heard ang   target's probability 0.1%


uvm       leng3    heard ang   target's probability 0.1%


uvm       zheng3   heard ang   target's probability 0.1%


uvm       teng3    heard ang   target's probability 0.1%


...

uvm       cheng2   heard ang   target's probability 7.4%


uvm       heng2    heard ang   target's probability 8.0%


uvm       deng2    heard ang   target's probability 14.2%


uvm       leng2    heard ang   target's probability 18.7%


uvm       meng2    heard ang   target's probability 34.7%


In [15]:
import os
print(os.path.exists("/content/std/standard/uvm"), os.path.exists("/content/tone_perfect"))

True True
